# CNN + LSTM — Optuna Tuning

## Imports

In [ ]:
import optuna
import numpy as np
import torch
import torch.nn as nn
import os
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

work_dir = os.getcwd()
print(work_dir)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Data Loader

In [ ]:
input_folder = os.path.join(work_dir, "data_cnn_normalized")

X_train = np.load(os.path.join(input_folder, "X_train.npy"))
y_train = np.load(os.path.join(input_folder, "y_train.npy"))
X_val = np.load(os.path.join(input_folder, "X_val.npy"))
y_val = np.load(os.path.join(input_folder, "y_val.npy"))
X_test = np.load(os.path.join(input_folder, "X_test.npy"))
y_test = np.load(os.path.join(input_folder, "y_test.npy"))

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"y_train: [{y_train.min():.2f}, {y_train.max():.2f}]")


def create_loader(X, y, batch_size=128, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)


train_loader = create_loader(X_train, y_train)
val_loader = create_loader(X_val, y_val, shuffle=False)
test_loader = create_loader(X_test, y_test, shuffle=False)


## Attention

In [ ]:
class Attention(nn.Module):
    """Additive (Bahdanau-style) attention over the time axis.
    Input : (B, T, H)   Output: (B, H)
    Replaces the last-hidden-state bottleneck so no timestep is forgotten.
    """

    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, x):  # x: (B, T, H)
        score = self.v(torch.tanh(self.attn(x)))  # (B, T, 1)
        weight = torch.softmax(score, dim=1)  # (B, T, 1)
        return (weight * x).sum(dim=1)  # (B, H)


## Objective

In [ ]:
def objective(trial):
    base_ch = trial.suggest_categorical("base_ch", [16, 32, 64])
    lstm_hidden = trial.suggest_int("lstm_hidden", 64, 256, step=32)
    lstm_layers = trial.suggest_int("lstm_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    huber_delta = trial.suggest_float("huber_delta", 1.0, 5.0)

    class _CNN_LSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.cnn = nn.Sequential(
                nn.Conv1d(X_train.shape[2], base_ch, 7, padding=3),
                nn.BatchNorm1d(base_ch),
                nn.ReLU(),
                nn.Conv1d(base_ch, base_ch * 2, 5, padding=2),
                nn.BatchNorm1d(base_ch * 2),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(base_ch * 2, base_ch * 2, 3, padding=1),
                nn.BatchNorm1d(base_ch * 2),
                nn.ReLU(),
            )
            self.lstm = nn.LSTM(
                base_ch * 2,
                lstm_hidden,
                lstm_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if lstm_layers > 1 else 0.0,
            )
            self.attn = Attention(lstm_hidden * 2)
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(lstm_hidden * 2, lstm_hidden),
                nn.ReLU(),
                nn.Linear(lstm_hidden, 1),
            )

        def forward(self, x):
            x = x.permute(0, 2, 1)
            x = self.cnn(x)
            x = x.permute(0, 2, 1)
            out, _ = self.lstm(x)
            return self.head(self.attn(out)).squeeze(-1)

    _train = create_loader(X_train, y_train, batch_size)
    _val = create_loader(X_val, y_val, batch_size, shuffle=False)

    m = _CNN_LSTM().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.HuberLoss(delta=huber_delta)
    best_val = float("inf")

    for epoch in range(40):
        m.train()
        for xb, yb in _train:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            crit(m(xb), yb).backward()
            opt.step()
        m.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in _val:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl.append(crit(m(xb), yb).item())
        val_loss = np.mean(vl)
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        if val_loss < best_val:
            best_val = val_loss
    return best_val


## Study

In [ ]:
study = optuna.create_study(
    direction="minimize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study.optimize(objective, n_trials=50)

## Results

In [ ]:
print("Best Loss  :", study.best_value)
print("Best Params:", study.best_params)
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout()
plt.show()
optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout()
plt.show()